In [1]:
import pandas as pd
import numpy as np
import os
import json
import ast

In [2]:
file_path = r'C:\dev\projects\NLP_test\data\train_dataset_target_as_json.csv'
data = pd.read_csv(file_path)
data.describe()

,max(page),target,available,target_original,target_conversion_review
count,201,201,201,79,201
unique,201,77,2,76,2
top,https://www.factorybuys.com.au/products/euro-t...,[],False,Gift Card,False
freq,1,122,105,4,171


In [3]:
def parse_targets(raw):
    if pd.isna(raw):
        return []

    s = str(raw).strip()
    if not s:
        return []

    for parser in (json.loads, ast.literal_eval):
        try:
            obj = parser(s)
            if isinstance(obj, list):
                return [str(x).strip() for x in obj if str(x).strip()]
        except Exception:
            pass

    return [s]

data["targets"] = data["target"].apply(parse_targets)
data["has_product"] = data["targets"].map(bool)

In [4]:
data.head(10)

,max(page),target,available,target_original,target_conversion_review,targets,has_product
0,https://www.factorybuys.com.au/products/euro-t...,"[""Factory Buys 32cm Euro Top Mattress - King""]",True,Factory Buys 32cm Euro Top Mattress - King,False,[Factory Buys 32cm Euro Top Mattress - King],True
1,https://dunlin.com.au/products/beadlight-cirrus,"[""Beadlight Cirrus LED Reading Light""]",True,Beadlight Cirrus LED Reading Light,False,[Beadlight Cirrus LED Reading Light],True
2,https://themodern.net.au/products/hamar-plant-...,"[""Hamar Plant Stand - Ash""]",True,Hamar Plant Stand - Ash,False,[Hamar Plant Stand - Ash],True
3,https://furniturefetish.com.au/products/oslo-o...,[],True,NaN,False,[],False
4,https://hemisphereliving.com.au/products/,[],False,NaN,False,[],False
5,https://home-buy.com.au/products/bridger-penda...,[],False,NaN,False,[],False
6,https://interiorsonline.com.au/products/interi...,"[""$50 RJ Living Gift Card"", ""$100 RJ Living Gi...",True,"$50 RJ Living Gift Card, $100 RJ Living Gift C...",True,"[$50 RJ Living Gift Card, $100 RJ Living Gift ...",True
7,https://beckurbanfurniture.com.au/products/pag...,[],False,NaN,False,[],False
8,https://livingedge.com.au/products/tables/dining,"[""Linear Wood Table"", ""Saarinen Dining Table, ...",True,"Linear Wood Table, Saarinen Dining Table, Oval...",True,"[Linear Wood Table, Saarinen Dining Table, Ova...",True
9,https://edenliving.online/collections/summerlo...,[],False,NaN,False,[],False


In [5]:
train_data = data[data["available"] == True].copy()

In [6]:
train_data.describe()

,max(page),target,available,target_original,target_conversion_review,targets,has_product
count,96,96,96,78,96,96,96
unique,96,76,1,75,2,76,2
top,https://www.factorybuys.com.au/products/euro-t...,[],True,Gift Card,False,[],True
freq,1,18,96,4,67,18,78


In [7]:
from sklearn.model_selection import train_test_split

X = train_data["max(page)"].tolist()
y = train_data["targets"].tolist()
stratify_labels = train_data["has_product"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42, stratify = stratify_labels)

print(f"train: {len(X_train)}, test: {len(X_test)}")
print(f"train has_product share: {sum(bool(x) for x in y_train) / len(y_train):.3f}")
print(f"test has_product share: {sum(bool(x) for x in y_test) / len(y_test):.3f}")

train: 67, test: 29
train has_product share: 0.806
test has_product share: 0.828


In [8]:
import re
import json
import random
import hashlib
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup, Tag

TIMEOUT = 30

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

_SKIP_TAGS = {"header", "footer", "aside", "nav"}
_SKIP_ATTRS = re.compile(
    r"\b(nav|footer|header|menu|breadcrumb|sidebar|filter|facet|pagination|"
    r"cart|checkout|newsletter|popup|modal|cookie|banner|subscribe|"
    r"social|share|wishlist-modal|refinement)\b",
    re.IGNORECASE,
)

_BAD_TEXT_RE = re.compile(
    r"\b(cookie|privacy|policy|terms|sign in|log in|subscribe|newsletter|"
    r"javascript|wishlist|compare|share|sort by|filter by|view more|load more)\b",
    re.IGNORECASE,
)

_CARD_PATTERN = re.compile(
    r"\b(product[-_]?card|product[-_]?tile|product[-_]?item|"
    r"grid[-_]?item|collection[-_]?item|plp[-_]item|"
    r"ProductCard|productCard)\b",
    re.IGNORECASE,
)

_PRODUCT_URL_PATTERNS = [
    re.compile(r"/[A-Z0-9][\w-]*\.html$", re.IGNORECASE),
    re.compile(r"/products/[\w-]+$", re.IGNORECASE),
    re.compile(r"/product/[\w-]+/?$", re.IGNORECASE),
    re.compile(r"/(?:p|item|dp)/[\w-]+/?$", re.IGNORECASE),
]

_TEXT_TAGS = ("h1", "h2", "h3", "p", "li", "span", "div")
_TOKEN_RE = re.compile(r"[A-Za-zА-Яа-я0-9]+(?:[-_/][A-Za-zА-Яа-я0-9]+)*")


def _clean(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def _tokenize(text: str) -> list[str]:
    return _TOKEN_RE.findall(text)


def _token_count(text: str) -> int:
    return len(_tokenize(text))


def _trim_to_tokens(text: str, max_tokens: int) -> str:
    if max_tokens <= 0:
        return ""
    toks = _tokenize(text)
    return " ".join(toks[:max_tokens])


def _is_price(text: str) -> bool:
    text = text.strip()
    return bool(
        re.match(
            r"^[\$€£¥₹]?\s?\d[\d,.\s]*(?:[\$€£¥₹]|usd|eur|byn|rub|pln)?$",
            text,
            re.I,
        )
    )


def _is_noise(text: str) -> bool:
    text = _clean(text)
    if not text:
        return True
    if len(text) < 3:
        return True
    if _BAD_TEXT_RE.search(text):
        return True
    if _is_price(text):
        return True
    if re.fullmatch(r"[\W_]+", text):
        return True
    if sum(ch.isalpha() for ch in text) < 2:
        return True
    return False


def _in_skip_zone(tag: Tag) -> bool:
    for parent in [tag, *tag.parents]:
        if not isinstance(parent, Tag):
            continue
        if parent.name in _SKIP_TAGS:
            return True
        cls = " ".join(parent.get("class", []))
        id_ = parent.get("id", "")
        role = parent.get("role", "")
        check = f"{cls} {id_} {role}"
        if _SKIP_ATTRS.search(check):
            return True
    return False


def _fetch(url: str):
    try:
        response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        if response.status_code == 200:
            return BeautifulSoup(response.text, "html.parser"), response.url, response.status_code
        return None, response.url, response.status_code
    except Exception:
        return None, url, 0


def _dedupe_keep_order(items: list[str]) -> list[str]:
    seen = set()
    out = []
    for x in items:
        key = x.casefold()
        if key in seen:
            continue
        seen.add(key)
        out.append(x)
    return out


def _walk_ld(node, out: list[str]):
    if isinstance(node, list):
        for item in node:
            _walk_ld(item, out)
        return

    if not isinstance(node, dict):
        return

    types = node.get("@type", "")
    types = types if isinstance(types, list) else [types]

    if any(t in ("Product", "IndividualProduct") for t in types):
        name = _clean(node.get("name", ""))
        if name and not _is_noise(name):
            out.append(name)
        return

    if "ItemList" in types:
        for elem in node.get("itemListElement", []):
            _walk_ld(elem, out)

    if "@graph" in node:
        _walk_ld(node["@graph"], out)

    for key, val in node.items():
        if key.startswith("@"):
            continue
        if isinstance(val, (dict, list)):
            _walk_ld(val, out)


def _from_jsonld(soup: BeautifulSoup) -> list[str]:
    products = []
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string or "")
        except Exception:
            continue
        _walk_ld(data, products)
    return products


def _looks_like_product_url(href: str, base_host: str) -> bool:
    parsed = urlparse(href)
    if parsed.netloc and parsed.netloc != base_host:
        return False
    return any(pat.search(parsed.path) for pat in _PRODUCT_URL_PATTERNS)


def _name_from_img_link(a_tag: Tag):
    img = a_tag.find("img")
    if not img:
        return None
    for attr in ("title", "alt"):
        val = _clean(img.get(attr, "") or "")
        if val and not _is_noise(val) and not _is_price(val):
            return val
    return None


def _name_from_text_link(a_tag: Tag):
    lines = [_clean(x) for x in a_tag.get_text("\n").split("\n") if _clean(x)]
    for line in lines:
        if not _is_noise(line) and not _is_price(line):
            return line
    return None


def _from_product_links(soup: BeautifulSoup, base_url: str) -> list[str]:
    base_host = urlparse(base_url).netloc
    found = {}

    for a in soup.find_all("a", href=True):
        href = urljoin(base_url, a["href"])
        norm = urlparse(href)._replace(query="", fragment="").geturl()

        if not _looks_like_product_url(href, base_host):
            continue
        if _in_skip_zone(a):
            continue
        if norm in found:
            continue

        name = _name_from_img_link(a) or _name_from_text_link(a)
        if name:
            found[norm] = name

    return list(found.values())


def _name_from_card(card: Tag):
    for level in ("h1", "h2", "h3", "h4", "h5"):
        h = card.find(level)
        if h:
            txt = _clean(h.get_text())
            if txt and not _is_noise(txt) and not _is_price(txt):
                return txt

    for el in card.find_all(True):
        cls = " ".join(el.get("class", []))
        if re.search(r"(product|item|card)[_-]?(name|title)", cls, re.I):
            txt = _clean(el.get_text())
            if txt and not _is_noise(txt) and not _is_price(txt):
                return txt

    a = card.find("a")
    if a:
        return _name_from_text_link(a)

    return None


def _from_html_cards(soup: BeautifulSoup) -> list[str]:
    products = []
    seen = set()

    candidates = [
        tag for tag in soup.find_all(True)
        if _CARD_PATTERN.search(" ".join(tag.get("class", [])))
    ]

    filtered = []
    for c in candidates:
        if not any(c in f.descendants for f in filtered):
            filtered = [f for f in filtered if f not in c.descendants]
            filtered.append(c)

    for card in filtered:
        if _in_skip_zone(card):
            continue
        name = _name_from_card(card)
        if name and name.casefold() not in seen:
            seen.add(name.casefold())
            products.append(name)

    return products


def _from_title(soup: BeautifulSoup):
    title_tag = soup.find("title")
    if title_tag:
        raw = _clean(title_tag.get_text())
        txt = re.split(r"\s*[|–—]\s*", raw)[0].strip()
        txt = re.sub(r"\s*[-–]\s*(Shop|Store|Buy|Online).*$", "", txt, flags=re.I).strip()
        if txt and not _is_noise(txt):
            return txt
    return None


def _detect_page_type(products: list[str]) -> str:
    return "catalog" if len(products) >= 6 else "single"


def _collect_noise_candidates(soup: BeautifulSoup, products: list[str], page_type: str) -> list[str]:
    product_keys = {p.casefold() for p in products}
    seen = set()
    out = []

    for tag in soup.find_all(_TEXT_TAGS):
        if _in_skip_zone(tag):
            continue
        if tag.find(_TEXT_TAGS):
            continue

        txt = _clean(tag.get_text(" ", strip=True))
        if _is_noise(txt):
            continue
        if txt.casefold() in product_keys:
            continue

        n = _token_count(txt)
        max_piece = 12 if page_type == "single" else 16

        if n > max_piece:
            txt = _trim_to_tokens(txt, max_piece)
            n = _token_count(txt)

        if n < 2:
            continue

        key = txt.casefold()
        if key in seen:
            continue
        seen.add(key)
        out.append(txt)

    return out


def _take_noise_by_budget(candidates: list[str], max_tokens: int) -> list[str]:
    picked = []
    cur = 0

    for txt in candidates:
        if cur >= max_tokens:
            break

        n = _token_count(txt)
        if n <= 0:
            continue

        if cur + n <= max_tokens:
            picked.append(txt)
            cur += n
        else:
            remain = max_tokens - cur
            clipped = _trim_to_tokens(txt, remain)
            if clipped:
                picked.append(clipped)
            break

    return picked


def _stable_seed(text: str) -> int:
    return int(hashlib.sha256(text.encode("utf-8")).hexdigest()[:8], 16)


def scrape_some_page_info(
    url: str,
    include_status: bool = False,
    shuffle_blocks: bool = False,
) -> str:
    soup, final_url, status_code = _fetch(url)
    if not soup:
        return f"status_{status_code}" if include_status else ""

    products = _dedupe_keep_order(
        _from_jsonld(soup) +
        _from_html_cards(soup) +
        _from_product_links(soup, final_url)
    )

    title = _from_title(soup)
    if title and title.casefold() not in {p.casefold() for p in products}:
        products = [title] + products

    page_type = _detect_page_type(products) if products else "single"
    noise_candidates = _collect_noise_candidates(soup, products, page_type)

    noise_budget = 120 if products else 80
    noise = _take_noise_by_budget(noise_candidates, max_tokens=noise_budget)

    blocks = []

    if include_status:
        blocks.append(f"status_{status_code}")

    if products:
        blocks.extend(products[:40])
    blocks.extend(noise)

    blocks = _dedupe_keep_order([_clean(x) for x in blocks if _clean(x)])

    if shuffle_blocks:
        rnd = random.Random(_stable_seed(url))
        rnd.shuffle(blocks)

    return "\n".join(blocks)


In [9]:
X_train_texts = [scrape_some_page_info(url) for url in X_train]

In [10]:
from huggingface_hub import login
login()

In [11]:
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased", use_fast = True)

In [12]:
import unicodedata

_TRANSLATION = str.maketrans({
    "–": "-",
    "—": "-",
    "−": "-",
    "’": "'",
    "‘": "'",
    "“": '"',
    "”": '"',
})


def normalize_for_matching(text: str):
    text = unicodedata.normalize("NFKC", str(text)).translate(_TRANSLATION)

    out = []
    idx_map = []
    prev_space = True

    for i, ch in enumerate(text):
        ch = " " if ch.isspace() else ch

        if ch == " " and prev_space:
            continue

        prev_space = (ch == " ")
        out.append(ch.lower())
        idx_map.append(i)

    if out and out[-1] == " ":
        out.pop()
        idx_map.pop()

    return "".join(out), idx_map


def build_target_pattern(target_norm: str) -> str:
    pat = re.escape(target_norm)

    if target_norm and target_norm[0].isalnum():
        pat = r"(?<!\w)" + pat
    if target_norm and target_norm[-1].isalnum():
        pat = pat + r"(?!\w)"

    return pat


def find_target_positions(text: str, targets: list[str]):
    if not text or not targets:
        return []

    norm_text, idx_map = normalize_for_matching(text)
    spans = []

    for target in targets:
        if not target:
            continue

        norm_target, _ = normalize_for_matching(target)
        if not norm_target:
            continue

        pattern = build_target_pattern(norm_target)

        for m in re.finditer(pattern, norm_text, flags=re.IGNORECASE):
            orig_start = idx_map[m.start()]
            orig_end = idx_map[m.end() - 1] + 1
            spans.append((orig_start, orig_end, target))

    spans.sort(key=lambda x: (x[0], x[1]))
    return spans

In [13]:
def build_encodings(texts, targets_list, tokenizer, max_length = 256, stride = 64):
    encodings = tokenizer(texts, truncation = True, padding = "max_length", max_length = max_length,
            stride = stride, return_overflowing_tokens = True, return_offsets_mapping = True, return_tensors = "pt")

    sample_map = encodings.pop("overflow_to_sample_mapping")
    offset_mapping = encodings.pop("offset_mapping").tolist()

    all_labels = []

    for chunk_idx, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[chunk_idx].item()
        text = texts[sample_idx]
        targets = targets_list[sample_idx]

        target_positions = find_target_positions(text, targets)

        chunk_labels = []
        for start, end in offsets:
            if start == 0 and end == 0:
                chunk_labels.append(-100)
                continue

            is_target = any(
                max(start, t_start) < min(end, t_end)
                for t_start, t_end, _ in target_positions
            )
            chunk_labels.append(1 if is_target else 0)

        all_labels.append(chunk_labels)

    encodings["labels"] = torch.tensor(all_labels, dtype=torch.long)
    return encodings


train_encodings = build_encodings(X_train_texts, y_train, tokenizer, max_length = 256, stride = 64)

In [14]:
from torch.utils.data import Dataset

class NERDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = {
            key: torch.tensor(val) if not torch.is_tensor(val) else val
            for key, val in encodings.items()
        }

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings["input_ids"])

train_dataset = NERDataset(train_encodings)

In [15]:
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [16]:
from transformers import BertForTokenClassification

id2label = {0: "NO_PRODUCT", 1: "PRODUCT"}
label2id = {"NO_PRODUCT": 0, "PRODUCT": 1}

model = BertForTokenClassification.from_pretrained("bert-base-cased", num_labels=2,
                                    id2label=id2label, label2id=label2id)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [17]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

X_test_texts = [scrape_some_page_info(url) for url in X_test]
test_encodings = build_encodings(X_test_texts, y_test, tokenizer, max_length = 256, stride = 64)

test_dataset = NERDataset(test_encodings)

train_loader = DataLoader(train_dataset, batch_size = 4, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 4, shuffle = False)

optimizer = AdamW(model.parameters(), lr=3e-5)

class_weights = torch.tensor([1.0, 6.0], dtype=torch.float, device = device)
loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)

In [18]:
from tqdm.auto import tqdm

epochs = 5

for epoch in range(epochs):
    model.train()
    loop = tqdm(train_loader, leave=True)
    epoch_loss = 0.0

    for batch in loop:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        loss = loss_fct(logits.view(-1, 2), labels.view(-1))
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} finished. Average Loss: {epoch_loss / len(train_loader):.4f}")

model.save_pretrained(r"C:\dev\projects\NLP_test\res\final_model")

  0%|          | 0/22 [00:00<?, ?it/s]

Epoch 1 finished. Average Loss: 0.5796


  0%|          | 0/22 [00:00<?, ?it/s]

Epoch 2 finished. Average Loss: 0.2837


  0%|          | 0/22 [00:00<?, ?it/s]

...

  0%|          | 0/22 [00:00<?, ?it/s]

Epoch 4 finished. Average Loss: 0.0907


  0%|          | 0/22 [00:00<?, ?it/s]

Epoch 5 finished. Average Loss: 0.0581


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
model.config.id2label = id2label
model.config.label2id = label2id

model.save_pretrained(r"C:\dev\projects\NLP_test\res\final_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [20]:
def evaluate_model(model, data_loader, dataset_name="Target Set"):
    model.eval()
    model.to(device)

    total_loss = 0.0
    all_predictions = []
    all_true_labels = []

    print(f"Evaluating on {dataset_name}...")

    with torch.no_grad():
        for batch in tqdm(data_loader):
            labels = batch["labels"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            loss = loss_fct(logits.view(-1, 2), labels.view(-1))
            predictions = torch.argmax(logits, dim=-1)

            total_loss += loss.item()
            all_predictions.extend(predictions.cpu().numpy())
            all_true_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(data_loader)
    print(f"{dataset_name} Loss: {avg_loss:.4f}")

    return avg_loss, all_true_labels, all_predictions

In [21]:
from sklearn.metrics import classification_report

def get_classification_report(model, data_loader, dataset_name="Target Set"):
    label_list = ["NO_PRODUCT", "PRODUCT"]
    model.eval()
    model.to(device)

    flat_predictions = []
    flat_true_labels = []

    print(f"Calculating metrics on {dataset_name}...")

    with torch.no_grad():
        for batch in tqdm(data_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"]

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions = torch.argmax(outputs.logits, dim=-1).cpu()

            for i in range(labels.shape[0]):
                for j in range(labels.shape[1]):
                    gold = labels[i, j].item()
                    if gold == -100:
                        continue

                    pred = predictions[i, j].item()
                    flat_true_labels.append(label_list[gold])
                    flat_predictions.append(label_list[pred])

    print(f"\n{dataset_name} Classification Report")
    print(classification_report(flat_true_labels, flat_predictions, zero_division=0))

In [22]:
train_loss, train_true, train_pred = evaluate_model(model, train_loader, "Train")

Evaluating on Train...


  0%|          | 0/22 [00:00<?, ?it/s]

Train Loss: 0.0455


In [23]:
get_classification_report(model, train_loader, "Train")

Calculating metrics on Train...


  0%|          | 0/22 [00:00<?, ?it/s]


Train Classification Report
              precision    recall  f1-score   support

  NO_PRODUCT       1.00      0.97      0.99      9902
     PRODUCT       0.90      1.00      0.95      2568

    accuracy                           0.98     12470
   macro avg       0.95      0.99      0.97     12470
weighted avg       0.98      0.98      0.98     12470



In [24]:
test_encodings["input_ids"].shape, test_encodings["labels"].shape

(torch.Size([31, 256]), torch.Size([31, 256]))

In [25]:
test_dataset = NERDataset(test_encodings)
test_loader = DataLoader(test_dataset, batch_size = 4, shuffle = False)

In [26]:
test_loss, test_true, test_pred = evaluate_model(model, test_loader, "Test")

Evaluating on Test...


  0%|          | 0/8 [00:00<?, ?it/s]

Test Loss: 0.4112


In [27]:
get_classification_report(model, test_loader, "Test")

Calculating metrics on Test...


  0%|          | 0/8 [00:00<?, ?it/s]


Test Classification Report
              precision    recall  f1-score   support

  NO_PRODUCT       0.95      0.86      0.90      3155
     PRODUCT       0.57      0.80      0.66       712

    accuracy                           0.85      3867
   macro avg       0.76      0.83      0.78      3867
weighted avg       0.88      0.85      0.86      3867

